# LightAD — Log Anomaly Detection Benchmark

**Paper:** Yu et al. (ICSE 2024) — *Deep Learning or Classical Machine Learning? An Empirical Study on Log-Based Anomaly Detection*  
**Models:** KNN · Decision Tree · SLFN (MLP)  
**Datasets:** HDFS_v1 · BGL  

---
### Kaggle Setup
1. Upload datasets vào Kaggle Dataset `logad-data`:
   - `HDFS.log` → `/kaggle/input/logad-data/hdfs/HDFS.log`
   - `anomaly_label.csv` → `/kaggle/input/logad-data/hdfs/anomaly_label.csv`
   - `BGL.log` → `/kaggle/input/logad-data/bgl/BGL.log`
2. GPU không cần thiết (LightAD chạy tốt trên CPU)
3. Run All

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/rubyhcm/code-papers.git'
REPO_DIR  = '/kaggle/working/code-papers'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready:', REPO_DIR)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q scikit-learn bayesian-optimization numpy

# distance package (may fail on newer Python — handled below)
!pip install -q distance || echo 'distance install failed (non-critical)'

In [ ]:
# ── 3. Setup paths & symlinks ─────────────────────────────────────────────────
import os, sys
import numpy as np

LIGHTAD_DIR = os.path.join(REPO_DIR, 'LightAD')
sys.path.insert(0, LIGHTAD_DIR)

# Data paths (Kaggle input)
HDFS_LOG    = '/kaggle/input/logad-data/hdfs/HDFS.log'
HDFS_LABEL  = '/kaggle/input/logad-data/hdfs/anomaly_label.csv'
BGL_LOG     = '/kaggle/input/logad-data/bgl/BGL.log'
OUTPUT_DIR  = '/kaggle/working/lightad_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LightAD reads data from datasets/original_datasets/<name>
ORIG_DIR = os.path.join(LIGHTAD_DIR, 'datasets', 'original_datasets')
os.makedirs(ORIG_DIR, exist_ok=True)

# Symlink raw data files
for src, dst in [
    (HDFS_LOG,   os.path.join(ORIG_DIR, 'hdfs')),
    (HDFS_LABEL, os.path.join(ORIG_DIR, 'anomaly_label.csv')),
    (BGL_LOG,    os.path.join(ORIG_DIR, 'bgl')),
]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked: {dst}')

print('Paths OK')

## Preprocessing

In [ ]:
# ── 4. Preprocess HDFS ────────────────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python preprocess.py --dataset hdfs

hdfs_split_dir = os.path.join(LIGHTAD_DIR, 'datasets', 'splited_datasets', 'hdfs')
print('HDFS splits:', os.listdir(hdfs_split_dir))

In [ ]:
# ── 5. Preprocess BGL ─────────────────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python preprocess.py --dataset bgl

bgl_split_dir = os.path.join(LIGHTAD_DIR, 'datasets', 'splited_datasets', 'bgl')
print('BGL splits:', os.listdir(bgl_split_dir))

## Train & Evaluate — HDFS

In [ ]:
# ── 6a. HDFS — KNN ────────────────────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python main_hdfs.py --model knn 2>&1 | tee {OUTPUT_DIR}/hdfs_knn.log

# Parse result
with open(f'{OUTPUT_DIR}/hdfs_knn.log') as f:
    for line in f:
        if 'Current Model' in line:
            print('RESULT:', line.strip())

In [ ]:
# ── 6b. HDFS — Decision Tree ──────────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python main_hdfs.py --model dt 2>&1 | tee {OUTPUT_DIR}/hdfs_dt.log

with open(f'{OUTPUT_DIR}/hdfs_dt.log') as f:
    for line in f:
        if 'Current Model' in line:
            print('RESULT:', line.strip())

In [ ]:
# ── 6c. HDFS — SLFN ───────────────────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python main_hdfs.py --model slfn 2>&1 | tee {OUTPUT_DIR}/hdfs_slfn.log

with open(f'{OUTPUT_DIR}/hdfs_slfn.log') as f:
    for line in f:
        if 'Current Model' in line:
            print('RESULT:', line.strip())

## Train & Evaluate — BGL

In [ ]:
# ── 7. BGL — KNN (semantic match) ─────────────────────────────────────────────
os.chdir(LIGHTAD_DIR)
!python main_super.py --dataset bgl 2>&1 | tee {OUTPUT_DIR}/bgl_knn.log

with open(f'{OUTPUT_DIR}/bgl_knn.log') as f:
    for line in f:
        if 'Currently Used Dataset' in line:
            print('RESULT:', line.strip())

## Collect & Save Results

In [ ]:
# ── 8. Parse all log files → results dict ─────────────────────────────────────
import re, json
import matplotlib.pyplot as plt
import numpy as np

def parse_hdfs_log(path):
    """Extract avg metrics from LightAD HDFS log."""
    result = {}
    with open(path) as f:
        for line in f:
            m = re.search(
                r'Current Model: (\w+);.*Precision: ([\d.]+).*Recall: ([\d.]+).*F1-score: ([\d.]+)',
                line)
            if m:
                result = dict(
                    model=m.group(1),
                    precision=float(m.group(2)),
                    recall=float(m.group(3)),
                    f1=float(m.group(4)))
            # train/infer time
            t = re.search(r'Train time: ([\d.]+)', line)
            if t: result['train_time_s'] = float(t.group(1))
            i = re.search(r'Inference time: ([\d.]+)', line)
            if i: result['infer_time_s'] = float(i.group(1))
    return result

def parse_bgl_log(path):
    result = {}
    with open(path) as f:
        for line in f:
            m = re.search(
                r'Currently Used Dataset: (\w+);.*Precision: ([\d.]+).*Recall: ([\d.]+).*F1-score: ([\d.]+)',
                line)
            if m:
                result = dict(
                    dataset=m.group(1),
                    precision=float(m.group(2)),
                    recall=float(m.group(3)),
                    f1=float(m.group(4)))
    return result

results = {
    'metadata': {'model': 'LightAD', 'paper': 'ICSE 2024'},
    'hdfs': {
        'knn':  parse_hdfs_log(f'{OUTPUT_DIR}/hdfs_knn.log'),
        'dt':   parse_hdfs_log(f'{OUTPUT_DIR}/hdfs_dt.log'),
        'slfn': parse_hdfs_log(f'{OUTPUT_DIR}/hdfs_slfn.log'),
    },
    'bgl': {
        'knn': parse_bgl_log(f'{OUTPUT_DIR}/bgl_knn.log'),
    },
}

out_path = f'{OUTPUT_DIR}/lightad_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

## Visualize

In [ ]:
# ── 9. Plot results ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('LightAD Results', fontsize=14, fontweight='bold')
metrics   = ['precision', 'recall', 'f1']
colors    = ['#2196F3', '#4CAF50', '#FF9800']
x         = np.arange(len(metrics))
width     = 0.25

# HDFS
ax = axes[0]
for i, (mname, vals) in enumerate(results['hdfs'].items()):
    if vals:
        bars = [vals.get(m, 0) for m in metrics]
        ax.bar(x + i*width, bars, width, label=mname.upper(), color=colors[i], alpha=0.85)
        for j, v in enumerate(bars):
            ax.text(j + i*width, v + 0.0001, f'{v:.4f}', ha='center', va='bottom', fontsize=7)
ax.set_title('HDFS_v1', fontweight='bold')
ax.set_xticks(x + width); ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0.98, 1.001); ax.legend(); ax.grid(axis='y', alpha=0.3)

# BGL
ax = axes[1]
bgl = results['bgl'].get('knn', {})
if bgl:
    bars = [bgl.get(m, 0) for m in metrics]
    ax.bar(x, bars, 0.5, color='#2196F3', alpha=0.85, label='KNN')
    for j, v in enumerate(bars):
        ax.text(j, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('BGL', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0.95, 1.01); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/lightad_results.png', bbox_inches='tight')
plt.show()
print(f'Results saved to {OUTPUT_DIR}')